In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cellpose import models
from cellpose.io import imread
import os
import pandas as pd

In [ ]:
images_dir = "/scr/jcaicedo/Micronuclei-data/dataset_v2/"
out_dir = images_dir + "experiments/2023-11-08/predictions/"

file_names = [x for x in os.listdir(images_dir) if x.endswith(".phenotype.tif")]
input_files = [f"{images_dir}/{x}" for x in file_names]
output_files = [out_dir + x.replace(".phenotype.tif","-nuclei.csv") for x in file_names]

In [ ]:
imgs = [imread(f) for f in input_files]
nimg = len(imgs)

In [ ]:
model = models.Cellpose(model_type='nuclei')
channels = [0,0]

In [ ]:
masks, flows, styles, diams = model.eval(imgs, diameter=None, channels=channels)

In [ ]:
def collect_stats(mask, img):
    data = []
    for i in range(1,len(np.unique(mask))):
        ys,xs = np.where(mask == i)
        intensity = np.mean(img[ys,xs])
        a,b = int(np.mean(ys)), int(np.mean(xs))
        area = np.sum(mask == i)
        data.append({"Image":files[0], "x":b, "y":a, "area":area, "intensity": intensity})

    return pd.DataFrame(data)

In [ ]:
for i in range(nimg):
    print(file_names[i])
    df = collect_stats(masks[i], imgs[i])
    df.to_csv(output_files[i])